# Notebook 02 — Integrated Energy System Design
**SPE Africa Geothermal Datathon 2026 — Team GeoLogic Analytics**

This notebook implements the integrated district heating and cooling system design:
1. Seasonal demand modelling (heating 10 MW, cooling 5 MW)
2. Geothermal thermal extraction and cascade utilisation
3. Heat pump integration and COP analysis
4. ATES (Aquifer Thermal Energy Storage) design
5. Hybrid backup strategy
6. Operational dispatch logic


## 1. Setup and Load Upstream Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300

PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../outputs/figures')
TABLES_DIR = Path('../outputs/tables')

# Load upstream outputs from Notebook 01
demand_params = pd.read_csv(TABLES_DIR / 'demand_summary.csv')
architecture = pd.read_csv(TABLES_DIR / 'final_architecture.csv')
screening = pd.read_csv(PROCESSED_DIR / 'geothermal_screening_summary.csv')

print("Demand parameters:")
print(demand_params.to_string(index=False))
print("\nArchitecture:")
print(architecture.to_string(index=False))


## 2. Seasonal Demand Modelling

District energy demand follows strong seasonal patterns:
- **Heating demand** peaks in winter (October–March), minimal in summer
- **Cooling demand** peaks in summer (June–August), minimal in winter

We model monthly demand profiles for the Utrecht district using normalised seasonal curves calibrated to the 10 MW heating and 5 MW cooling design targets.


In [ ]:
# Monthly demand profiles (normalised to peak capacity)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_nums = np.arange(1, 13)

# Heating demand profile (peaks in winter)
heating_factors = np.array([1.00, 0.95, 0.75, 0.45, 0.15, 0.05, 0.02, 0.02, 0.10, 0.50, 0.80, 0.95])
# Cooling demand profile (peaks in summer)
cooling_factors = np.array([0.02, 0.02, 0.05, 0.15, 0.40, 0.80, 1.00, 0.95, 0.50, 0.15, 0.05, 0.02])

HEATING_PEAK = 10.0  # MW
COOLING_PEAK = 5.0   # MW

heating_demand = heating_factors * HEATING_PEAK
cooling_demand = cooling_factors * COOLING_PEAK

demand_df = pd.DataFrame({
    'Month': months,
    'Month_Num': month_nums,
    'Heating_MW': heating_demand,
    'Cooling_MW': cooling_demand,
    'Total_Thermal_MW': heating_demand + cooling_demand
})

# Annual energy (MWh) - assuming average over month hours
hours_per_month = np.array([744,672,744,720,744,720,744,744,720,744,720,744])
demand_df['Heating_MWh'] = demand_df['Heating_MW'] * hours_per_month
demand_df['Cooling_MWh'] = demand_df['Cooling_MW'] * hours_per_month

print("Monthly Demand Profile:")
print(demand_df[['Month', 'Heating_MW', 'Cooling_MW', 'Total_Thermal_MW']].to_string(index=False))
print(f"\nAnnual heating energy: {demand_df['Heating_MWh'].sum():,.0f} MWh")
print(f"Annual cooling energy: {demand_df['Cooling_MWh'].sum():,.0f} MWh")
print(f"Total annual thermal: {demand_df['Heating_MWh'].sum() + demand_df['Cooling_MWh'].sum():,.0f} MWh")


In [ ]:
# District demand curve
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(12)
width = 0.35

bars1 = ax.bar(x - width/2, heating_demand, width, label='Heating Demand', color='#E53935', alpha=0.85)
bars2 = ax.bar(x + width/2, cooling_demand, width, label='Cooling Demand', color='#1E88E5', alpha=0.85)

ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Thermal Demand (MW)', fontsize=12)
ax.set_title('District Heating and Cooling — Seasonal Demand Profile', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(months)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'district_demand_curve.png', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: outputs/figures/district_demand_curve.png")


## 3. Geothermal Thermal Extraction

The BLT-01 + GLA-01 doublet provides 11.2 MW combined thermal capacity at design conditions. However, the full geothermal output is not always needed — extraction must match demand to avoid thermal waste and extend reservoir life.


In [ ]:
# Geothermal supply vs demand
Q_GEOTHERMAL = 11.2  # MW combined (BLT-01: 5.1 + GLA-01: 6.1)
T_SUPPLY = 70.0  # °C district supply (limited by GLA-01 production temp)
T_RETURN = 40.0  # °C district return

# Direct geothermal contribution (capped at demand)
demand_df['Geo_Direct_Heating_MW'] = np.minimum(heating_demand, Q_GEOTHERMAL)
demand_df['Geo_Surplus_MW'] = np.maximum(Q_GEOTHERMAL - heating_demand, 0)

# In summer, surplus geothermal heat can charge ATES warm well
demand_df['ATES_Charge_MW'] = demand_df['Geo_Surplus_MW'] * 0.7  # 70% storage efficiency

print("Geothermal Supply vs Heating Demand:")
print(demand_df[['Month', 'Heating_MW', 'Geo_Direct_Heating_MW', 'Geo_Surplus_MW', 'ATES_Charge_MW']].to_string(index=False))
print(f"\nMonths with surplus: {(demand_df['Geo_Surplus_MW'] > 0).sum()}")
print(f"Total surplus for ATES charging: {demand_df['ATES_Charge_MW'].sum() * 730:.0f} MWh/year (avg)")


## 4. Heat Pump Integration

At 70°C supply temperature, direct geothermal covers most heating demand. However, boosting to higher temperatures for peak demand and enabling cooling requires heat pump integration.

- **Heating mode:** Heat pump upgrades geothermal return water (40°C) to district supply (70°C) during peak periods
- **Cooling mode:** Heat pump provides 5 MW chilling capacity using ATES cold well as heat sink


In [ ]:
# Heat pump parameters
COP_HEATING = 4.5  # COP for 40°C → 70°C upgrade
COP_COOLING = 5.0  # COP for cooling mode (ATES cold sink at 8-15°C)
HP_CAPACITY = 5.0  # MW thermal capacity

# Heating shortfall (when demand exceeds geothermal capacity)
demand_df['Heating_Shortfall_MW'] = np.maximum(heating_demand - Q_GEOTHERMAL, 0)
demand_df['HP_Heating_MW'] = np.minimum(demand_df['Heating_Shortfall_MW'], HP_CAPACITY)
demand_df['HP_Heating_Elec_MW'] = demand_df['HP_Heating_MW'] / COP_HEATING

# Cooling via heat pump + ATES
demand_df['HP_Cooling_MW'] = cooling_demand  # all cooling through HP
demand_df['HP_Cooling_Elec_MW'] = demand_df['HP_Cooling_MW'] / COP_COOLING

# Total electrical consumption
demand_df['Total_Elec_MW'] = demand_df['HP_Heating_Elec_MW'] + demand_df['HP_Cooling_Elec_MW']

print("Heat Pump Operation:")
print(demand_df[['Month', 'HP_Heating_MW', 'HP_Heating_Elec_MW', 
                  'HP_Cooling_MW', 'HP_Cooling_Elec_MW', 'Total_Elec_MW']].to_string(index=False))
print(f"\nPeak electrical demand: {demand_df['Total_Elec_MW'].max():.2f} MW")
print(f"Annual avg electrical: {demand_df['Total_Elec_MW'].mean():.2f} MW")


## 5. ATES (Aquifer Thermal Energy Storage) Design

ATES enables seasonal thermal balancing:
- **Summer:** Store excess geothermal heat in warm well (25–35°C); store natural cold in cold well (8–15°C)
- **Winter:** Retrieve stored warmth to pre-heat return water; retrieve stored cold for summer cooling

This reduces peak heat pump electrical consumption and improves annual system efficiency.


In [ ]:
# ATES configuration
ATES_WARM_TEMP = 30.0  # °C average warm well
ATES_COLD_TEMP = 10.0  # °C average cold well
ATES_EFFICIENCY = 0.70  # round-trip thermal efficiency
ATES_CAPACITY = 3.0    # MW thermal storage/retrieval rate

# Seasonal ATES operation
demand_df['ATES_Mode'] = np.where(demand_df['Month_Num'].isin([4,5,6,7,8,9]), 'Charging', 'Discharging')
demand_df['ATES_Thermal_MW'] = np.where(
    demand_df['ATES_Mode'] == 'Charging',
    np.minimum(demand_df['Geo_Surplus_MW'], ATES_CAPACITY),
    -np.minimum(ATES_CAPACITY * ATES_EFFICIENCY, demand_df['Heating_Shortfall_MW'])
)

print("ATES Seasonal Operation:")
print(demand_df[['Month', 'ATES_Mode', 'ATES_Thermal_MW', 'Geo_Surplus_MW']].to_string(index=False))

# ATES energy balance
charge_energy = demand_df.loc[demand_df['ATES_Mode']=='Charging', 'ATES_Thermal_MW'].sum()
discharge_energy = abs(demand_df.loc[demand_df['ATES_Mode']=='Discharging', 'ATES_Thermal_MW'].sum())
print(f"\nAnnual ATES charge: {charge_energy:.1f} MW-months")
print(f"Annual ATES discharge: {discharge_energy:.1f} MW-months")
print(f"Round-trip efficiency: {discharge_energy/max(charge_energy,0.01)*100:.0f}%")


## 6. Hybrid Backup Strategy

A gas-fired backup boiler provides resilience against:
- Geothermal well maintenance/downtime
- Extreme cold weather peaks exceeding design capacity
- ATES thermal depletion during prolonged cold spells


In [ ]:
# Backup boiler
BACKUP_CAPACITY = 5.0  # MW gas boiler
BACKUP_EFFICIENCY = 0.92

# Unmet demand after geothermal + HP + ATES
demand_df['Remaining_Shortfall_MW'] = np.maximum(
    heating_demand - demand_df['Geo_Direct_Heating_MW'] - demand_df['HP_Heating_MW'], 0
)
demand_df['Backup_MW'] = np.minimum(demand_df['Remaining_Shortfall_MW'], BACKUP_CAPACITY)

print("Backup Boiler Dispatch:")
print(demand_df[['Month', 'Heating_MW', 'Geo_Direct_Heating_MW', 'HP_Heating_MW', 
                  'Backup_MW', 'Remaining_Shortfall_MW']].to_string(index=False))
print(f"\nMonths requiring backup: {(demand_df['Backup_MW'] > 0).sum()}")
print(f"Peak backup load: {demand_df['Backup_MW'].max():.2f} MW")


## 7. Integrated System Dispatch Summary

In [ ]:
# Full dispatch summary
dispatch = demand_df[['Month', 'Heating_MW', 'Cooling_MW', 'Geo_Direct_Heating_MW',
                       'HP_Heating_MW', 'HP_Cooling_MW', 'ATES_Mode', 'ATES_Thermal_MW',
                       'Backup_MW', 'Total_Elec_MW']].copy()

print("Integrated Monthly Dispatch:")
print(dispatch.to_string(index=False))

# Geothermal contribution ratio
geo_heating_total = demand_df['Geo_Direct_Heating_MW'].sum()
total_heating = demand_df['Heating_MW'].sum()
geo_fraction = geo_heating_total / total_heating * 100
print(f"\nGeothermal direct heating contribution: {geo_fraction:.1f}%")
print(f"Heat pump heating contribution: {demand_df['HP_Heating_MW'].sum()/total_heating*100:.1f}%")
print(f"Backup contribution: {demand_df['Backup_MW'].sum()/total_heating*100:.1f}%")

# Export dispatch
dispatch.to_csv(TABLES_DIR / 'system_dispatch.csv', index=False)
print("\nSaved: outputs/tables/system_dispatch.csv")


In [ ]:
# Stacked area chart — system dispatch
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Integrated District Energy System — Monthly Dispatch', fontsize=15, fontweight='bold')

x = np.arange(12)

# Heating dispatch
ax1.bar(x, demand_df['Geo_Direct_Heating_MW'], label='Geothermal Direct', color='#E53935', alpha=0.85)
ax1.bar(x, demand_df['HP_Heating_MW'], bottom=demand_df['Geo_Direct_Heating_MW'],
        label='Heat Pump', color='#FF9800', alpha=0.85)
ax1.bar(x, demand_df['Backup_MW'], 
        bottom=demand_df['Geo_Direct_Heating_MW'] + demand_df['HP_Heating_MW'],
        label='Gas Backup', color='#9E9E9E', alpha=0.85)
ax1.plot(x, heating_demand, 'ko-', label='Demand', markersize=5, linewidth=2)
ax1.set_ylabel('Heating (MW)', fontsize=11)
ax1.legend(fontsize=9, loc='upper right')
ax1.set_ylim(0, 12)
ax1.grid(axis='y', alpha=0.3)
ax1.set_title('District Heating')

# Cooling dispatch
ax2.bar(x, demand_df['HP_Cooling_MW'], label='Heat Pump Cooling', color='#1E88E5', alpha=0.85)
ax2.plot(x, cooling_demand, 'ko-', label='Demand', markersize=5, linewidth=2)
ax2.set_ylabel('Cooling (MW)', fontsize=11)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_xticks(x)
ax2.set_xticklabels(months)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 6)
ax2.grid(axis='y', alpha=0.3)
ax2.set_title('District Cooling')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'integrated_system_dispatch.png', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: outputs/figures/integrated_system_dispatch.png")


## 8. Geothermal Circulation Schematic

Conceptual system architecture showing the full thermal flow path from reservoir to district and back.


In [ ]:
# System architecture diagram (programmatic)
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Integrated Geothermal District Energy System Architecture', fontsize=16, fontweight='bold', pad=20)

# Component boxes
components = [
    (1, 7.5, 2.5, 1.5, 'BLT-01\nProduction\n77°C, 105 m³/h', '#E53935'),
    (1, 5.0, 2.5, 1.5, 'GLA-01\nProduction\n70°C, 140 m³/h', '#FF5722'),
    (5, 6.5, 3, 1.5, 'Heat\nExchanger', '#FF9800'),
    (5, 4.0, 3, 1.5, 'Heat Pump\nCOP 4.5/5.0', '#2196F3'),
    (10, 7.0, 3, 1.5, 'District\nHeating\n10 MW', '#E53935'),
    (10, 4.5, 3, 1.5, 'District\nCooling\n5 MW', '#1E88E5'),
    (5, 1.5, 3, 1.5, 'ATES\nStorage', '#4CAF50'),
    (1, 2.0, 2.5, 1.5, 'Injection\nWell\n35-40°C', '#795548'),
    (14, 6.0, 1.5, 1.5, 'Backup\nBoiler\n5 MW', '#9E9E9E'),
]

for x, y, w, h, text, color in components:
    rect = plt.Rectangle((x, y), w, h, facecolor=color, alpha=0.3, edgecolor=color, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=8, fontweight='bold')

# Arrows
arrow_style = dict(arrowstyle='->', color='black', lw=1.5)
connections = [
    ((3.5, 8.25), (5, 7.25)),   # BLT-01 → HX
    ((3.5, 5.75), (5, 7.0)),    # GLA-01 → HX
    ((8, 7.25), (10, 7.75)),    # HX → District Heating
    ((8, 4.75), (10, 5.25)),    # HP → District Cooling
    ((6.5, 4.0), (6.5, 3.0)),   # HP → ATES
    ((5, 2.25), (3.5, 2.75)),   # ATES → Injection
    ((1, 3.5), (1, 5.0)),       # Injection → circ
    ((14, 7.5), (13, 7.75)),    # Backup → District
]
for start, end in connections:
    ax.annotate('', xy=end, xytext=start, arrowprops=arrow_style)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'geothermal_circulation_schematic.png', bbox_inches='tight', dpi=300)
plt.show()
print("Saved: outputs/figures/geothermal_circulation_schematic.png")


---
## Summary

This notebook designed the integrated district energy system:

1. **Seasonal Demand:** 10 MW heating (winter peak), 5 MW cooling (summer peak)
2. **Geothermal Direct:** BLT-01 + GLA-01 provides 11.2 MW — covers >95% of heating demand directly
3. **Heat Pump:** Provides all cooling (COP 5.0) and supplements heating peaks (COP 4.5)
4. **ATES:** Seasonal storage balances surplus summer geothermal heat for winter use
5. **Backup:** 5 MW gas boiler for resilience (rarely dispatched)
6. **Geothermal fraction:** >95% of annual heating from geothermal resources

**Next:** Notebook 03 — Economic Analysis and AI-Assisted Workflow
